# Step 2: Conflict detection under controlled sensor noise

Each noise type is applied to the clean SinD trajectories at several strengths; velocity and
heading are then re-derived from the noisy positions (as a camera pipeline would), the full
conflict screen from step 1 is rerun, and the result is compared with the clean reference.

**Long runs:** use the script, which can run tasks in parallel and is safe to leave overnight:
`python scripts/03_noise_experiments.py --data-root ... --jobs 4`. Finished tasks are cached,
so this notebook can rebuild the tables at any time with `collect_results`.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
from IPython.display import Image, display
from sensors_noise.noise_pipeline import run_noise_experiments, collect_results

DATA_ROOT = Path(r"PATH\TO\sind")
OUT_DIR = Path.cwd().parent / "results" / "02_noise"

## Option A: quick run from the notebook (one seed, one level per type; serial, slower)

Skip this if you ran the script.

In [ ]:
QUICK_GRID = {"baseline": [0.0], "jitter": [0.3, 1.0], "distance_jitter": [0.5], "drift": [0.5],
              "offset": [0.6], "heading": [5.0], "dropout": [0.15], "id_switch": [1.0]}
res = run_noise_experiments(DATA_ROOT, str(OUT_DIR) + "_quick", grid=QUICK_GRID, seeds=(0,))

## Option B: load results of a finished (or partly finished) script run

In [ ]:
res = collect_results(OUT_DIR)

## Detection: recall (clean conflicts still found) and precision (found conflicts that are real)

Counts are averages per seed, pooled over sites.

In [ ]:
cols = ["n_clean", "found", "missed", "false", "recall", "precision", "severity_shift_rate"]
res["detection"][cols]

## Value errors on conflicts found in both (seconds; noisy minus clean)

In [ ]:
res["errors"]

## Where false conflicts come from (share involving a road user that is static in the reference)

In [ ]:
res.get("false_diagnostics")

## Per site

In [ ]:
res["detection_by_site"][cols]

## Figures

In [ ]:
for f in sorted(Path(OUT_DIR).glob("noise_*.png")) + sorted(Path(str(OUT_DIR) + "_quick").glob("noise_*.png")):
    print(f.name); display(Image(filename=str(f)))